<a href="https://colab.research.google.com/github/thiagoest/genai-news-classification/blob/main/fine_tuning_lm_head_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Bibliotecas
!pip install -q transformers datasets accelerate scikit-learn

ERROR: Operation cancelled by user


In [6]:
# Credencial de acesso ao HuggingFace
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

In [57]:
# Carregando Dataset
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import random

# Exemplo usando classificação de notícias em 4 categorias (ag_news usando o caminho completo usuario/nome_do_dataset)
dataset = load_dataset("fancyzhx/ag_news")
NUM_CLASSES = 4
MODEL_ID = "Qwen/Qwen2.5-0.5B"  # Decoder

In [58]:
# Estrutura do Dataset
print("Schema e qtde de registros \n")
dataset

Schema e qtde de registros 



DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [59]:
# Conteúdo do Dataset

label_names = {
    0: "world",
    1: "sports",
    2: "business",
    3: "sci/tech",
}

print("Alguns exemplos de notícias \n")

random_sample = random.sample(range(0, 120000), 5)
for i in random_sample:
    print(f"Registro {i}")
    print("Texto:", dataset["train"][i]["text"])
    print("Label:", dataset["train"][i]["label"], "(", label_names[dataset["train"][i]["label"]], ")")
    print()

Alguns exemplos de notícias 

Registro 6846
Texto: Four Dead After Attack on Homeless Man Four homeless men were bludgeoned to death and six were in critical condition on Friday following early morning attacks by unknown assailants in downtown streets of Sao Paulo, a police spokesman said.
Label: 0 ( world )

Registro 45072
Texto: Tax Benefits Lift PepsiCo Profit PepsiCo Inc. (PEP.N: Quote, Profile, Research) , the world #39;s No. 2 soft drink company, said on Thursday quarterly profit rose due to tax benefits and a strong performance from its key 
Label: 2 ( business )

Registro 17480
Texto: Mids Need Motivation Coming off its third winning season in the last 20 years, Navy has looked sluggish on offense and uninspired overall during the last several weeks of practice.
Label: 1 ( sports )

Registro 30767
Texto: Jaguar puts brakes on at historic plant Production is to cease at a Midlands factory where some of the world #39;s most iconic cars were made. Mark Tran looks at its history.
L

In [69]:
# Tokenização

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

# Amostragem reduzida para testes
sample_size_train = 4000
sample_size_test = 800

train_ds = dataset["train"].shuffle(seed=42).select(range(sample_size_train)).map(tokenize_fn, batched=True)
test_ds = dataset["test"].shuffle(seed=42).select(range(sample_size_test)).map(tokenize_fn, batched=True)

In [61]:
## Verificando um exemplo
print(train_ds.column_names)
print("\nExemplo tokenizado:")
print(train_ds[random.randint(1,100)])

['text', 'label', 'input_ids', 'attention_mask']

Exemplo tokenizado:
{'text': 'West sets deadline for Iran to freeze uranium enrichment Four western countries set the scene yesterday for a showdown with Iran by demanding that it freeze its uranium enrichment activities immediately.', 'label': 0, 'input_ids': [23306, 7289, 21428, 369, 10259, 311, 29930, 53844, 69172, 13322, 18494, 5837, 738, 279, 6109, 13671, 369, 264, 71608, 448, 10259, 553, 25135, 429, 432, 29930, 1181, 53844, 69172, 7488, 7069, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 15164

In [62]:
# Configuração do pad_token e formato do Dataset (PyTorch)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32)

In [63]:
# Verificando um exemplo
#print(train_loader.column_names)
print("\nExemplo formatado p/ PyToch:")
print(train_loader.dataset[random.randint(1,100)])


Exemplo formatado p/ PyToch:
{'label': tensor(0), 'input_ids': tensor([ 61719,  46459,    547,    808,     13,    444,   7821,  14566,    389,
         11445,  30316,   6687,    320,  35878,      8,  22841,    481,   6515,
           702,   4201,    264,   5593,  13068,    916,     59,  61719,    594,
         11249,  52476,     11,  31061,   3322,  11064,    311,   7479,    547,
          2067,   7110,  33082,   5136,    323,   2238,  34419,   3556,   1495,
           304,  11093,    369,    264,   6277,  13169,  26266,  26137,  64452,
           730,   9198,  10606,   1053,   7728,     13, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643,
        151643, 151643, 151643, 151643, 151643, 151643, 

In [64]:
# Substituindo o LM Head

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Verifica se a GPU realmente suporta bfloat16; caso contrário, usa float32 para estabilidade no treino
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    model_dtype = torch.bfloat16
else:
    model_dtype = torch.float32

# Remove o LM Head original e cria a nova cabeça de classificação com NUM_CLASSES neurônios
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    torch_dtype=model_dtype
).to(device)

model.config.pad_token_id = tokenizer.pad_token_id

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [65]:
# Detalhes do modelo
print("Hidden size:", model.config.hidden_size)
print("Número de camadas:", model.config.num_hidden_layers)
print("Número de labels:", model.config.num_labels)
print("Tipo do modelo:", type(model).__name__)

Hidden size: 896
Número de camadas: 24
Número de labels: 4
Tipo do modelo: Qwen2ForSequenceClassification


In [66]:
# Especificação do experimento (Fine-Tuning vs. Transfer Learning)

CONGELAR_BACKBONE = False  # True: Transfer Learning; False: Fine-tuning

if CONGELAR_BACKBONE:
    # Congela todas as camadas exceto a cabeça de classificação ('classifier' ou 'score')
    for name, param in model.named_parameters():
        if "classifier" not in name and "score" not in name:
            param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros totais: {total_params:,}")
print(f"Parâmetros treináveis: {trainable_params:,}")

Parâmetros totais: 494,036,352
Parâmetros treináveis: 494,036,352


In [68]:
# Treinamento (Fine tuning)

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=2e-5,
    weight_decay=0.01
)

NUM_EPOCHS = 3

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    print(f"Época {epoch+1}/{NUM_EPOCHS} | Loss: {total_loss/len(train_loader):.4f} | Acurácia Treino: {correct/total:.4f}")

Época 1/3 | Loss: 0.0996 | Acurácia Treino: 0.9645
Época 2/3 | Loss: 0.0135 | Acurácia Treino: 0.9965
Época 3/3 | Loss: 0.0024 | Acurácia Treino: 1.0000


In [70]:
# Avaliando na amostra de validação

import torch
from sklearn.metrics import classification_report

def avaliar_modelo(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    todas_preds = []
    todos_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=-1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # .float() evita erro de conversão caso o modelo esteja em bfloat16
            todas_preds.extend(preds.float().cpu().numpy())
            todos_labels.extend(labels.float().cpu().numpy())

    loss_medio = total_loss / len(dataloader)
    acuracia = correct / total
    return loss_medio, acuracia, todos_labels, todas_preds

# Executando a comparação após o loop de treinamento
loss_treino, acc_treino, _, _ = avaliar_modelo(model, train_loader, device)
loss_teste, acc_teste, y_true, y_pred = avaliar_modelo(model, test_loader, device)

print("=== Comparativo Final: Treino vs. Teste ===")
print(f"Treino -> Loss: {loss_treino:.4f} | Acurácia: {acc_treino * 100:.2f}%")
print(f"Teste  -> Loss: {loss_teste:.4f} | Acurácia: {acc_teste * 100:.2f}%")
print(f"Diferença (Overfitting gap): {(acc_treino - acc_teste) * 100:.2f} p.p.\n")

print("=== Relatório Detalhado por Classe (Teste) ===")
print(classification_report(y_true, y_pred, digits=4))

=== Comparativo Final: Treino vs. Teste ===
Treino -> Loss: 0.0019 | Acurácia: 100.00%
Teste  -> Loss: 0.3828 | Acurácia: 90.38%
Diferença (Overfitting gap): 9.62 p.p.

=== Relatório Detalhado por Classe (Teste) ===
              precision    recall  f1-score   support

         0.0     0.9289    0.9059    0.9173       202
         1.0     0.9646    0.9896    0.9770       193
         2.0     0.8878    0.8488    0.8678       205
         3.0     0.8373    0.8750    0.8557       200

    accuracy                         0.9038       800
   macro avg     0.9047    0.9048    0.9045       800
weighted avg     0.9041    0.9038    0.9036       800

